In [11]:
import pandas as pd
import numpy as np

EPS = 1e-3  # sMAPE 안정화

def load_any(path):
    # utf-8 / cp949 호환 로더
    for enc in ["utf-8", "cp949"]:
        try: return pd.read_csv(path, encoding=enc)
        except: pass
    return pd.read_csv(path)

def to_long(df):
    cols = df.columns
    if ("영업일자" in cols) and ("영업장명_메뉴명" in cols) and ("매출수량" in cols):
        return df[["영업일자","영업장명_메뉴명","매출수량"]].copy()
    # wide → long
    date_col = next((c for c in cols if c=="영업일자" or "일자" in c or "date" in c.lower()), None)
    if date_col is None:
        df = df.copy(); df.insert(0, "영업일자", np.arange(len(df))); date_col = "영업일자"
    long = df.melt(id_vars=[date_col], var_name="영업장명_메뉴명", value_name="매출수량")
    long = long.rename(columns={date_col:"영업일자"})
    return long

def smape_pair(a, b, eps=EPS):
    num = (a - b).abs()
    den = (a.abs() + b.abs()).clip(lower=eps)
    return (2.0 * num / den)

# --- 경로만 바꿔서 실행 ---
path_attn = "./prediction/model_v9_25.csv"  # 새 모델(Attention)
path_base = "./Prediction/v8_01_nonuse_valid.csv"   # 기존 v7 or v8

p1 = to_long(load_any(path_attn)); p1 = p1.rename(columns={"매출수량":"pred_attn"})
p2 = to_long(load_any(path_base)); p2 = p2.rename(columns={"매출수량":"pred_base"})

m = pd.merge(p1, p2, on=["영업일자","영업장명_메뉴명"], how="inner")
m["smape_pair"] = smape_pair(m["pred_attn"], m["pred_base"])

print("Pairwise sMAPE (mean) =", m["smape_pair"].mean().round(4))
print("Pairwise sMAPE (p90)  =", m["smape_pair"].quantile(0.90).round(4))

# 어디서 차이가 큰지 TOP 리스트
top_rows = m.nlargest(30, "smape_pair")[["영업일자","영업장명_메뉴명","pred_attn","pred_base","smape_pair"]]
top_items = (m.groupby("영업장명_메뉴명")["smape_pair"].mean()
               .sort_values(ascending=False).head(30).reset_index())
display(top_rows); display(top_items)


Pairwise sMAPE (mean) = 0.5356
Pairwise sMAPE (p90)  = 1.2873


,영업일자,영업장명_메뉴명,pred_attn,pred_base,smape_pair
11119,TEST_08+4일,카페테리아_오픈푸드,307.684082,1.000000,1.987042
11868,TEST_05+4일,포레스트릿_떡볶이,1.000000,285.756561,1.986051
11121,TEST_08+6일,카페테리아_오픈푸드,282.531189,1.000000,1.985892
11120,TEST_08+5일,카페테리아_오픈푸드,250.868378,1.000000,1.984119
11876,TEST_06+5일,포레스트릿_떡볶이,1.000000,248.628135,1.983976
11118,TEST_08+3일,카페테리아_오픈푸드,238.115036,1.000000,1.983272
11875,TEST_06+4일,포레스트릿_떡볶이,1.000000,224.345251,1.982249
9266,TEST_03+6일,연회장_Regular Coffee,200.654007,1.000000,1.980164
9242,TEST_00+3일,연회장_Regular Coffee,198.006012,1.000000,1.979900
11117,TEST_08+2일,카페테리아_오픈푸드,190.343704,1.000000,1.979095


,영업장명_메뉴명,smape_pair
0,연회장_Cass Beer,1.876759
1,담하_(단체) 은이버섯 갈비탕,1.827102
2,담하_(단체) 한우 우거지 국밥,1.700589
3,연회장_Regular Coffee,1.582897
4,카페테리아_아메리카노(HOT),1.579888
5,미라시아_브런치(대인) 주말,1.177966
6,느티나무 셀프BBQ_BBQ55(단체),1.177648
7,느티나무 셀프BBQ_카스 병(단체),1.177538
8,카페테리아_단체식 13000(신),1.112651
9,담하_(단체) 생목살 김치전골 2.0,1.108698
